In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Stream the dataset directly into a pandas DataFrame
url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
df = pd.read_csv(url, sep='\t', header=None, names=['label', 'text'])

# Map string labels to numeric binaries: ham = 0, spam = 1
df['label_num'] = df['label'].map({'ham': 0, 'spam': 1})

# View the first 5 rows to ensure it loaded correctly
df.head()

,label,text,label_num
0,ham,"Go until jurong point, crazy.. Available only ...",0
1,ham,Ok lar... Joking wif u oni...,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,ham,U dun say so early hor... U c already then say...,0
4,ham,"Nah I don't think he goes to usf, he lives aro...",0


In [2]:
X = df['text']
y = df['label_num']

# Split: 80% for training parameters, 20% reserved for testing evaluation
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training sentences: {len(X_train)}")
print(f"Testing sentences: {len(X_test)}")

Training sentences: 4457
Testing sentences: 1115


In [3]:
# Initialize vectorizer and remove standard English stop words
vectorizer = TfidfVectorizer(stop_words='english')

# IMPORTANT: "Fit and transform" on training data, but ONLY "transform" test data
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print(f"Vocabulary vocabulary size extracted: {X_train_vec.shape[1]} unique words.")

Vocabulary vocabulary size extracted: 7403 unique words.


In [4]:
# Initialize models
nb_classifier = MultinomialNB()
lr_classifier = LogisticRegression(solver='liblinear')

# Train both models using our vector matrices
nb_classifier.fit(X_train_vec, y_train)
lr_classifier.fit(X_train_vec, y_train)

print("Both models have completed training successfully!")

Both models have completed training successfully!


In [5]:
# Run test predictions
nb_predictions = nb_classifier.predict(X_test_vec)
lr_predictions = lr_classifier.predict(X_test_vec)

print("="*20 + " NAIVE BAYES REPORT " + "="*20)
print(f"Overall Accuracy: {accuracy_score(y_test, nb_predictions):.4f}")
print(classification_report(y_test, nb_predictions, target_names=['Ham (Valid)', 'Spam']))

print("\n" + "="*20 + " LOGISTIC REGRESSION REPORT " + "="*20)
print(f"Overall Accuracy: {accuracy_score(y_test, lr_predictions):.4f}")
print(classification_report(y_test, lr_predictions, target_names=['Ham (Valid)', 'Spam']))

==================== NAIVE BAYES REPORT ====================
Overall Accuracy: 0.9704
              precision    recall  f1-score   support

 Ham (Valid)       0.97      1.00      0.98       966
        Spam       1.00      0.78      0.88       149

    accuracy                           0.97      1115
   macro avg       0.98      0.89      0.93      1115
weighted avg       0.97      0.97      0.97      1115


==================== LOGISTIC REGRESSION REPORT ====================
Overall Accuracy: 0.9677
              precision    recall  f1-score   support

 Ham (Valid)       0.96      1.00      0.98       966
        Spam       1.00      0.76      0.86       149

    accuracy                           0.97      1115
   macro avg       0.98      0.88      0.92      1115
weighted avg       0.97      0.97      0.97      1115



In [6]:
def test_new_message(custom_text):
    # Vectorize the raw input text using our fitted vocabulary pipeline
    vec_text = vectorizer.transform([custom_text])

    # Predict probabilities and classifications
    nb_pred = nb_classifier.predict(vec_text)[0]
    lr_pred = lr_classifier.predict(vec_text)[0]

    label_map = {0: "🟢 HAM (Legit)", 1: "🚨 SPAM"}

    print(f"Message: \"{custom_text}\"")
    print(f" -> Naive Bayes Choice: {label_map[nb_pred]}")
    print(f" -> Logistic Regression Choice: {label_map[lr_pred]}\n")

# Run some test examples
test_new_message("Hey, are we still meeting up for lunch at 1 PM today?")
test_new_message("URGENT! Your mobile number has won a £2,000 cash prize! Call 09051234567 now to claim your reward.")

Message: "Hey, are we still meeting up for lunch at 1 PM today?"
 -> Naive Bayes Choice: 🟢 HAM (Legit)
 -> Logistic Regression Choice: 🟢 HAM (Legit)

Message: "URGENT! Your mobile number has won a £2,000 cash prize! Call 09051234567 now to claim your reward."
 -> Naive Bayes Choice: 🚨 SPAM
 -> Logistic Regression Choice: 🚨 SPAM

